# Wikidata genre tree — exploration

Exploratory notebook over the Bronze `wikidata_genre_tree.parquet` output (see [`../SCHEMA.md`](../SCHEMA.md#bronze)) and the Silver `1_item_links`/`2_regional_overview_classification`/`3_regional_classification`/`4_genre_parents`/`5_hierarchy` outputs (see [`../SCHEMA.md`](../SCHEMA.md#silver)). Reads local Parquet files only — no live SPARQL calls.

**Prerequisite:** run the ingest and silver steps first so the Parquet files exist:

```bash
uv run --package wikidata python -m wikidata.ingest
uv run --package wikidata python -m wikidata.silver
```

## P31 vs. P279/P361 — how the genre set and the hierarchy are built

Two separate questions, two separate properties (see [`../SCHEMA.md#wikidata-properties-used`](../SCHEMA.md#wikidata-properties-used)):

- **"What is the full set of genres?"** → `P31` ("instance of"). `wd:Q11399` "rock music" `wdt:P31` `wd:Q188451` "music genre" — this class-membership edge is how Bronze finds all ~6,300 genre items (`GENRE_TREE_QUERY`'s `?item wdt:P31 wd:Q188451`).
- **"How do two genres relate?"** → `P279` ("subclass of") or `P361` ("part of"), *between genre items*. `wd:Q11399` "rock music" `wdt:P279` `wd:Q373342` "popular music" is the hierarchy edge.

`P31` is **never** used to build hierarchy edges, and `P279`/`P361` are **never** used to find the genre set — a direct `?item wdt:P279 wd:Q188451` query returns only 12 items, not real genres: one unrelated ("game piece"), two specific traditions ("gharana", "palo"), and seven meta-categories ("rock genre", "jazz genre", ...) describing a *category of genre* rather than the genre itself, because a genre's `P279` parent chain leads to broader *concepts* like "popular music", not back to the literal `Q188451` node. Confirmed live: `ASK { wd:Q373342 wdt:P279* wd:Q188451 }` → `false`, even though "popular music" is itself `P31`-classified as a music genre.

In [ ]:
import polars as pl
from common.env import load_pipeline_env, require_env, resolve_pipeline_path

import wikidata

load_pipeline_env(wikidata.__file__)

bronze_output_dir = resolve_pipeline_path(wikidata.__file__, require_env("BRONZE_OUTPUT_DIR"))
bronze_path = bronze_output_dir / "wikidata_genre_tree.parquet"
df = pl.read_parquet(bronze_path)
df.shape

## Tabular exploration

In [ ]:
df.head(10)

In [ ]:
df.group_by("relation_type").len().sort("len", descending=True)

In [ ]:
roots = df.filter(pl.col("parent_id").is_null())
print(f"{roots.height} root items")
roots.select("item_id", "item_label").head(10)

In [ ]:
multi_parent = (
    df.filter(pl.col("parent_id").is_not_null())
    .group_by("item_id", "item_label")
    .agg(pl.col("parent_id").n_unique().alias("n_parents"))
    .filter(pl.col("n_parents") > 1)
    .sort("n_parents", descending=True)
)
print(f"{multi_parent.height} items with more than one parent")
multi_parent.head(10)

## Silver exploration

### `1_item_links.parquet`
It carries the first Silver columns — `item_url`/`parent_url`, a browsable Wikidata page derived from `item_id`/`parent_id` — on top of the unchanged Bronze edge list.

In [ ]:
silver_output_dir = resolve_pipeline_path(wikidata.__file__, require_env("SILVER_OUTPUT_DIR"))

item_links_df = pl.read_parquet(silver_output_dir / "1_item_links.parquet")
item_links_df.select("item_id", "item_url", "parent_id", "parent_url").head(10)

### `2_regional_overview_classification.parquet`
It carries `is_regional_overview`/`classification_reason` on top of `1_item_links`.

In [ ]:
genre_class_df = pl.read_parquet(silver_output_dir / "2_regional_overview_classification.parquet")
genre_class_df.shape

In [ ]:
genre_class_df.group_by("is_regional_overview", "classification_reason").len().sort("len", descending=True)

In [ ]:
tagged = (
    genre_class_df.filter(pl.col("is_regional_overview"))
    .select("item_id", "item_label", "classification_reason")
    .unique()
)
print(f"{tagged.height} distinct items tagged regional-overview (not excluded — see 3_regional_classification below)")
tagged.head(10)

### 3_regional_classification — regional genre cascade

`3_regional_classification.parquet` adds two columns, `is_regional`/`regional_reason`, flagging which items are regional genres (see [`../SCHEMA.md#3_regional_classification`](../SCHEMA.md#3_regional_classification)). Flag-only — no rows dropped.

- **Seeds**: the `regional_overview` items from `2_regional_overview_classification` (e.g. "music of Cape Verde") start the cascade, and are themselves flagged `is_regional = true` / `regional_reason = "seed"` — regional genre nodes in their own right, not just a launching point.
- **Any-parent rule**: from those seeds, the flag cascades down to every descendant genre item — an item is regional if *any one* of its parent edges points to a seed or to an item already flagged regional.

**Exploration-phase finding**: on a real run this flags ~53% of all items (299 seed, 1,392 direct, 1,686 inherited), driven by continent-level seeds ("music of Asia", "music of Europe", ...) with large fan-out — kept as-is for now rather than narrowed, since the design is still exploratory.

In [ ]:
regional_df = pl.read_parquet(silver_output_dir / "3_regional_classification.parquet")
regional_df.shape

In [ ]:
regional_df.group_by("is_regional", "regional_reason").len().sort("len", descending=True).with_columns(
    (pl.col("len") / pl.col("len").sum() * 100).round(1).alias("pct")
)

In [ ]:
# music of Cape Verde is the seed itself; morna's only parent is that seed (direct); fado's
# parent is morna (inherited).
regional_df.filter(pl.col("item_label").is_in(["music of Cape Verde", "morna", "fado"])).select(
    "item_id", "item_label", "parent_id", "parent_label", "is_regional", "regional_reason"
).unique()

### 4_genre_parents

`4_genre_parents.parquet` adds `parent_is_genre`, flagging whether each edge's `parent_id` is itself flagged `is_regional_overview = False` by `2_regional_overview_classification` (see [`../SCHEMA.md#4_genre_parents`](../SCHEMA.md#4_genre_parents)). Flag-only, no rows dropped.

In [ ]:
genre_parents_df = pl.read_parquet(silver_output_dir / "4_genre_parents.parquet")
genre_parents_df.shape

In [ ]:
genre_parents_df.group_by("parent_is_genre").len().sort("len", descending=True)

In [ ]:
non_genre_parents = (
    genre_parents_df.filter(~pl.col("parent_is_genre"))
    .select("item_id", "item_label", "parent_id", "parent_label", "relation_type")
    .unique()
)
print(f"{non_genre_parents.height} edges into a non-genre parent")
non_genre_parents.head(10)

In [ ]:
canonical_genres_roots = genre_parents_df.filter(pl.col("parent_id").is_null() & ~pl.col("is_regional")).select(
    "item_id", "item_label"
)
print(f"{canonical_genres_roots.height} canonical roots")
with pl.Config(tbl_rows=-1):
    print(canonical_genres_roots)

### 5_hierarchy — pruned, single-parent-per-item (canonical)

The first Silver step that drops rows: filters to genre-only, non-regional edges, then collapses any item with more than one surviving parent down to the one with the lowest numeric QID — a **provisional heuristic** (see [`../SCHEMA.md#5_hierarchy`](../SCHEMA.md#5_hierarchy)), not a considered rule. This is the canonical output (`5_hierarchy.parquet`); regional items (`is_regional = true`) are excluded here and land in `5_regional_hierarchy.parquet` instead — see the section below.

In [ ]:
hierarchy_df = pl.read_parquet(silver_output_dir / "5_hierarchy.parquet")
hierarchy_df.shape

In [ ]:
genre_items = genre_parents_df.filter(~pl.col("is_regional_overview")).select("item_id").unique()
surviving_items = hierarchy_df.select("item_id").unique()
missing_from_canonical = genre_items.join(surviving_items, on="item_id", how="anti")
print(
    f"{missing_from_canonical.height} of {genre_items.height} genre items have zero rows in 5_hierarchy "
    "(either regional, routed to 5_regional_hierarchy instead, or every parent edge was non-genre)"
)
missing_from_canonical.head(10)

In [ ]:
# Sample of items with more than one genre-only parent candidate, and which one the lowest-QID
# heuristic kept (chosen_parent_id) vs. the alternatives it discarded.
multi_parent_candidates = genre_parents_df.filter(~pl.col("is_regional_overview") & pl.col("parent_is_genre")).select(
    "item_id", "item_label", "parent_id", "parent_label"
)
multi_parent_items = multi_parent_candidates.group_by("item_id").len().filter(pl.col("len") > 1).select("item_id")
print(f"{multi_parent_items.height} items have more than one genre parent")

sample = (
    multi_parent_candidates.join(multi_parent_items, on="item_id")
    .join(hierarchy_df.select("item_id", pl.col("parent_id").alias("chosen_parent_id")), on="item_id")
    .sort("item_id")
)
sample.head(10)

### Ruled out: `P279`-direct-subclass-of-`Q188451` as an alternate root/seed list

Prompted by the high canonical root count above, considered whether `?item wdt:P279 wd:Q188451`
(items directly subclass-of "music genre" itself, rather than `P31`-instance-of it) could serve
as a smaller, curated root/seed list instead of `5_hierarchy`'s hundreds of roots. Live Wikidata
returns only 12 items — not a usable top-level genre list:

| QID | label | why it doesn't work as a root |
| --- | --- | --- |
| Q3177825 | game piece | unrelated, likely a Wikidata miscategorization |
| Q1521426 | gharana | a specific Indian classical tradition, not a top-level root |
| Q1786517 | palo | a specific flamenco style, not a top-level root |
| Q20643324 | opera genre | meta-category ("a genre of opera"), not a genre item |
| Q21006644 | fusion music genre | meta-category |
| Q96195030 | jazz genre | meta-category — jazz music itself is the separate `P31` instance `Q1298934` |
| Q96200402 | electronic music genre | meta-category |
| Q105635529 | blues genre | meta-category |
| Q106621729 | folk music genre | meta-category |
| Q106628249 | world music genre | meta-category |
| Q107975727 | rock genre | meta-category |
| Q117021501 | music by instrument | meta-category, orthogonal grouping axis |

No prior art found (web search) for using this pattern to seed a Wikidata music genre tree.
Doesn't resolve the root-count question above — see [`../SCHEMA.md#5_hierarchy`](../SCHEMA.md#5_hierarchy)'s
"Under exploration" callout.

### 5_regional_hierarchy — pruned, single-parent-per-item (regional)

`5_regional_hierarchy.parquet` mirrors `5_hierarchy`'s pruning, restricted to `is_regional = true` items — which now includes the `regional_overview` seeds themselves as real nodes with real parent chains (or genuine roots, if a seed has no `P279`/`P361` parent of its own), rather than being dropped or promoted. An item like "morna," whose only parent is the "music of Cape Verde" seed, keeps that real parent edge instead of being promoted to a synthetic root (see [`../SCHEMA.md#5_hierarchy`](../SCHEMA.md#5_hierarchy)).

In [ ]:
regional_hierarchy_df = pl.read_parquet(silver_output_dir / "5_regional_hierarchy.parquet")
regional_hierarchy_df.shape

In [ ]:
# Every root here should be a regional_overview seed (regional_reason == "seed") — unlike the
# earlier design, non-seed regional items are never promoted to a synthetic root, since seeds
# are themselves real nodes with real parent chains.
roots = regional_hierarchy_df.filter(pl.col("parent_id").is_null()).select("item_id", "item_label")
root_reasons = roots.join(regional_df.select("item_id", "regional_reason").unique(), on="item_id", how="left")
print(f"{root_reasons.height} roots in 5_regional_hierarchy")
root_reasons.group_by("regional_reason").len().sort("len", descending=True)